# HyDE (Hypothetical Document Embeddings)

**HyDE (Hypothetical Document Embeddings)** is a **Query Translation** technique in Retrieval-Augmented Generation (RAG). Instead of embedding the user's original question, an LLM first generates a hypothetical answer (document). The embedding of this generated document is then used to retrieve relevant information from the vector database.

---

## Workflow

```text
User Question
      │
      ▼
Generate Hypothetical Document
      │
      ▼
Create Embedding
      │
      ▼
Retrieve Documents
      │
      ▼
Retrieved Context
      │
      ▼
LLM
      │
      ▼
Final Answer
```

---

## Advantages

- Improves retrieval for short or ambiguous queries.
- Captures the semantic meaning of the user's intent.
- Retrieves more relevant documents.
- Produces more grounded and accurate responses.

---

## One-Line Definition

> **HyDE is a Query Translation technique that generates a hypothetical document from the user's query, embeds it, and uses that embedding to retrieve relevant documents before generating the final answer.**


In [1]:
%run ./pipeline/01_basic_rag.ipynb
%run ./pipeline/02_indexing_rag.ipynb

1.3.13



C:\Users\Acer\AppData\Local\Temp\ipykernel_5580\3494673254.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
USER_AGENT environment variable not set, consider setting it to identify your requests.


Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss.
1.3.13
Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss.
384
384
Cosine Similarity: 0.7378822383225558


In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

# Local Ollama LLM use gareko
llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
)


# HyDE ko lagi hypothetical document generate garne prompt
template = """
Please write a scientific paper passage that answers the following question.

Question:
{question}

Passage:
"""

prompt_hyde = ChatPromptTemplate.from_template(template)


# HyDE document generation chain create gareko
generate_docs_for_retrieval = prompt_hyde | llm | StrOutputParser()


# User ko original question
question = "What is task decomposition for LLM agents?"


# Retrieval ko lagi hypothetical document generate gareko
hypothetical_document = generate_docs_for_retrieval.invoke(
    {
        "question": question,
    }
)


# Generate bhayeko hypothetical document print garne
print(hypothetical_document)

Here's a potential passage answering the question:

Task decomposition is a crucial component of Large Language Model (LLM) agent design, enabling these artificial intelligence systems to effectively tackle complex tasks by breaking them down into more manageable sub-tasks. In the context of LLMs, task decomposition involves identifying and isolating specific components or modules within a larger task that can be executed independently, leveraging the model's language understanding and generation capabilities.

By decomposing tasks in this manner, LLM agents can improve their overall performance by focusing on one aspect at a time, reducing the complexity and cognitive load associated with processing entire tasks. This approach also allows for more efficient learning and adaptation, as the agent can refine its skills in each sub-task before moving on to the next.

For instance, consider a conversational AI designed to assist customers with booking flights. A task decomposition strategy

In [3]:
# HyDE bata generate bhayeko hypothetical document use garera retrieval chain create gareko
retrieval_chain = generate_docs_for_retrieval | retriever


# Hypothetical document ko embedding use garera relevant documents retrieve gareko
retrieved_docs = retrieval_chain.invoke(
    {
        "question": question,
    }
)


# Retrieve bhayeka documents print garne
retrieved_docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='\n\n      LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistake

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Final answer generate garna RAG prompt create gareko
template = """
Answer the following question based on the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)


# Final HyDE RAG chain create gareko
final_rag_chain = (
    prompt
    # Local Ollama LLM bata final answer generate garne
    | llm
    # Output lai plain text ma convert garne
    | StrOutputParser()
)


# Retrieved context use garera final answer generate gareko
response = final_rag_chain.invoke(
    {
        "context": retrieved_docs,
        "question": question,
    }
)


# Final answer print garne
print(response)

According to the provided context, task decomposition for LLM (Large Language Model) agents involves breaking down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks. This allows the agent to plan and make progress towards achieving its goals.
